In [0]:
from pyspark.sql import functions as F

CATALOG = "civic_signal_dbx_dev"

RAW_VOLUME = f"/Volumes/{CATALOG}/bronze/raw_landing"
STATE_VOLUME = f"/Volumes/{CATALOG}/ops/autoloader_state"

In [0]:
opp_source = f"{RAW_VOLUME}/opportunities"

opp_schema = f"{STATE_VOLUME}/opportunities/schema"
opp_checkpoint = f"{STATE_VOLUME}/opportunities/checkpoint"

opp_target = f"{CATALOG}.bronze.opportunities_raw"

opp_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", opp_schema)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("rescuedDataColumn", "_rescued_data")
        .load(opp_source)
        .select(
            "*",
            F.current_timestamp().alias("_ingest_timestamp"),
            F.col("_metadata.file_path").alias("_source_file"),
            F.col("_metadata.file_modification_time")
             .alias("_source_file_modification_time")
        )
)

opp_query = (
    opp_df.writeStream
        .option("checkpointLocation", opp_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(opp_target)
)

opp_query.awaitTermination()

In [0]:
buyer_source = f"{RAW_VOLUME}/buyers"

buyer_schema = f"{STATE_VOLUME}/buyers/schema"
buyer_checkpoint = f"{STATE_VOLUME}/buyers/checkpoint"

buyer_target = f"{CATALOG}.bronze.buyers_raw"

buyer_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", buyer_schema)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("rescuedDataColumn", "_rescued_data")
        .load(buyer_source)
        .select(
            "*",
            F.current_timestamp().alias("_ingest_timestamp"),
            F.col("_metadata.file_path").alias("_source_file"),
            F.col("_metadata.file_modification_time")
             .alias("_source_file_modification_time")
        )
)

buyer_query = (
    buyer_df.writeStream
        .option("checkpointLocation", buyer_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(buyer_target)
)

buyer_query.awaitTermination()

In [0]:
category_source = f"{RAW_VOLUME}/categories/categories.csv"
category_target = f"{CATALOG}.bronze.categories_raw"

categories_df = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(category_source)
        .select(
            "*",
            F.current_timestamp().alias("_ingest_timestamp"),
            F.col("_metadata.file_path").alias("_source_file"),
            F.col("_metadata.file_modification_time")
             .alias("_source_file_modification_time")
        )
)

(
    categories_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(category_target)
)

In [0]:
spark.sql("""
COMMENT ON TABLE civic_signal_dbx_dev.bronze.opportunities_raw IS
'Raw synthetic procurement opportunities incrementally ingested from Civic Signal ADLS using Auto Loader. Source fields are preserved with ingestion and file metadata.'
""")

spark.sql("""
COMMENT ON TABLE civic_signal_dbx_dev.bronze.buyers_raw IS
'Raw synthetic public buyer records incrementally ingested from Civic Signal ADLS using Auto Loader.'
""")

spark.sql("""
COMMENT ON TABLE civic_signal_dbx_dev.bronze.categories_raw IS
'Raw synthetic procurement category reference data loaded from Civic Signal ADLS.'
""")

DataFrame[]

In [0]:
for table in [
    "opportunities_raw",
    "buyers_raw",
    "categories_raw"
]:
    count = spark.table(
        f"civic_signal_dbx_dev.bronze.{table}"
    ).count()

    print(f"{table}: {count}")

opportunities_raw: 13
buyers_raw: 7
categories_raw: 7
